In [20]:

team_colors = ["Red", "Yellow", "Green", "Orange", "Purple", "Pink", "Dark Blue", "Light Blue", "White", "Brown", "Gray", "Black"]


### Team
```json
{
    "Color": "String",
    "FirstGames": 0,
    "MiddleGames": 0,
    "LastGames": 0
}
```
### Game
```json
{
  "FirstTeams": [],
  "MiddleTeams": [],
  "LastTeams": [],
}
```

General Info
12 Teams Total
6 Weeks of Games
12 Games per Day

12 Choose 2 is 66 pairings
12 Choose 4 is 495 combos of 4 teams

(I think there's a dynamic way to do this?)

In [21]:
teams = {}

class Team:
    def __init__(self, team_color):
        self.color = team_color
        self.teams_played = set(team_color)
        self.pods_played = [0,0,0]

for color in team_colors:
    teams[color] = Team(color)


When scheduling a group of teams on a specific day there are a couple conditions to check:

- Have they played any of the teams in the pod previously?
- Has the team reached the max amount of times it can play in this time slot?
- Has the team already played today?

In [22]:
import itertools
import sys

possible_pods = set(itertools.combinations(team_colors, 4))
possible_matches= set(itertools.combinations(team_colors, 2))
len(possible_pods)
len(possible_matches)

def is_group_valid(possible_grouping, played_today):
    for team_color in possible_grouping:
        # Ensure teams haven't played before
        #Does the team_color team in the group have any of the other teams in its teams played?
        if len(set.intersection(teams[team_color].teams_played, set(possible_grouping))) > 1:
            return False
        #If this team plays at this time, will it exceed its pod play allowance?
        if teams[team_color].pods_played[pod] + 1 > 2:
            return False
        if team_color in played_today:
            return False
    return True

used_groupings = set()
schedule = [[ 0 for i in range(3)] for j in range(5)]
for week in range(5):
    played_today = set()
    for pod in range(3):

        while True:
            try:
                possible_grouping = possible_pods.pop()
            except KeyError:
                print("Ran out of pods")
                for w in range(week):
                    print("WEEK " + str(w + 1))
                    for p in range(pod):
                        print("POD " + str(pod + 1) + " : " + str(schedule[week][pod]))
                sys.exit()


            if is_group_valid(possible_grouping, played_today):
                break

        for team_color in possible_grouping:
            teams[team_color].teams_played.add(possible_grouping)
            teams[team_color].pods_played[pod] += 1
            played_today.add(team_color)

        schedule[week][pod] = possible_grouping
        print("Played today is: " + str(played_today))

for week in range(5):
    print("WEEK " + str(week + 1))
    for pod in range(3):
        print("POD " + str(pod + 1) + " : " + str(schedule[week][pod]))



Played today is: {'White', 'Pink', 'Orange', 'Dark Blue'}
Played today is: {'Gray', 'Pink', 'Orange', 'Yellow', 'Black', 'Dark Blue', 'Purple', 'White'}
Played today is: {'Gray', 'Pink', 'Orange', 'Yellow', 'Brown', 'Black', 'Green', 'Dark Blue', 'Red', 'Purple', 'Light Blue', 'White'}
Played today is: {'Gray', 'Brown', 'Pink', 'Dark Blue'}
Played today is: {'White', 'Gray', 'Pink', 'Yellow', 'Brown', 'Green', 'Light Blue', 'Dark Blue'}
Ran out of pods
WEEK 1
POD 3 : 0
POD 3 : 0


AttributeError: 'tuple' object has no attribute 'tb_frame'

In [23]:
schedule = [[ 0 for i in range(3)] for j in range(5)]

print(schedule)

schedule[0][0] = "hello"

print(schedule)


[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]
[['hello', 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]


In [46]:
from ortools.sat.python import cp_model

num_nurses = 12
num_shifts = 3
num_days = 6
all_nurses = range(num_nurses)
all_shifts = range(num_shifts)
all_days = range(num_days)

model = cp_model.CpModel()

shifts = {}
for n in all_nurses:
    for d in all_days:
        for s in all_shifts:
            shifts[(n, d, s)] = model.new_bool_var(f"shift_n{n}_d{d}_s{s}")

for d in all_days:
    for s in all_shifts:
        #model.add_exactly_one(shifts[(n, d, s)] for n in all_nurses)
        model.add(sum(shifts[(n, d, s)] for n in all_nurses) == 4)

for n in all_nurses:
    for d in all_days:
        #Check this later, may need to tweak
        model.add_at_most_one(shifts[(n, d, s)] for s in all_shifts)


min_shifts_per_nurse = (num_shifts * num_days) // num_nurses
if num_shifts * num_days % num_nurses == 0:
    max_shifts_per_nurse = min_shifts_per_nurse
else:
    max_shifts_per_nurse = min_shifts_per_nurse + 1

for n in all_nurses:
    shifts_worked = []
    for d in all_days:
        for s in all_shifts:
            shifts_worked.append(shifts[(n, d, s)])
    model.add(min_shifts_per_nurse <= sum(shifts_worked))
    model.add(sum(shifts_worked) <= max_shifts_per_nurse)




TypeError: add_linear_constraint() missing 2 required positional arguments: 'lb' and 'ub'

In [43]:
solver = cp_model.CpSolver()
solver.parameters.linearization_level = 0
solver.parameters.enumerate_all_solutions = True

In [44]:
class NursesPartialSolutionPrinter(cp_model.CpSolverSolutionCallback):
    """Print intermediate solutions."""

    def __init__(self, shifts, num_nurses, num_days, num_shifts, limit):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self._shifts = shifts
        self._num_nurses = num_nurses
        self._num_days = num_days
        self._num_shifts = num_shifts
        self._solution_count = 0
        self._solution_limit = limit

    def on_solution_callback(self):
        self._solution_count += 1
        print(f"Solution {self._solution_count}")
        for d in range(self._num_days):
            print(f"Day {d}")
            for n in range(self._num_nurses):
                is_working = False
                for s in range(self._num_shifts):
                    if self.value(self._shifts[(n, d, s)]):
                        is_working = True
                        print(f"  Nurse {n} works shift {s}")
                if not is_working:
                    print(f"  Nurse {n} does not work")
        if self._solution_count >= self._solution_limit:
            print(f"Stop search after {self._solution_limit} solutions")
            self.stop_search()

    def solutionCount(self):
        return self._solution_count

# Display the first five solutions.
solution_limit = 5
solution_printer = NursesPartialSolutionPrinter(
    shifts, num_nurses, num_days, num_shifts, solution_limit
)

In [45]:
solver.solve(model, solution_printer)

3

In [49]:
from ortools.sat.python import cp_model
from collections import defaultdict
import itertools

def solve_league_scheduling():
    """
    Solve the league scheduling problem using CP-SAT solver.

    Problem:
    - 12 teams, 3 pods (Early, Middle, Late) with 4 teams each
    - 6 weeks in season
    - Each team plays in exactly one pod per week
    - Each team plays every other team at least once
    - Each team plays in each pod at most twice
    """

    # Problem parameters
    num_teams = 12
    num_pods = 3
    teams_per_pod = 4
    num_weeks = 6

    teams = list(range(num_teams))
    pods = list(range(num_pods))  # 0=Early, 1=Middle, 2=Late
    weeks = list(range(num_weeks))

    # Create the model
    model = cp_model.CpModel()

    # Decision variables: x[team][week][pod] = 1 if team plays in pod during week
    x = {}
    for team in teams:
        for week in weeks:
            for pod in pods:
                x[team, week, pod] = model.NewBoolVar(f'x_{team}_{week}_{pod}')

    # Constraint 1: Each team plays in exactly one pod per week
    for team in teams:
        for week in weeks:
            model.Add(sum(x[team, week, pod] for pod in pods) == 1)

    # Constraint 2: Each pod has exactly 4 teams per week
    for week in weeks:
        for pod in pods:
            model.Add(sum(x[team, week, pod] for team in teams) == teams_per_pod)

    # Constraint 3: Each team plays in each pod at most twice
    for team in teams:
        for pod in pods:
            model.Add(sum(x[team, week, pod] for week in weeks) <= 2)

    # Constraint 4: Each team plays every other team at least once
    # Two teams play each other if they are in the same pod in the same week
    for team1 in teams:
        for team2 in teams:
            if team1 < team2:  # Avoid duplicate constraints
                # Create a variable for each week/pod combination where both teams could meet
                meeting_vars = []
                for week in weeks:
                    for pod in pods:
                        # Both teams are in the same pod in the same week
                        meet_var = model.NewBoolVar(f'meet_{team1}_{team2}_{week}_{pod}')
                        model.AddBoolAnd([x[team1, week, pod], x[team2, week, pod]]).OnlyEnforceIf(meet_var)
                        model.AddBoolOr([x[team1, week, pod].Not(), x[team2, week, pod].Not()]).OnlyEnforceIf(meet_var.Not())
                        meeting_vars.append(meet_var)

                # Teams must meet at least once
                model.Add(sum(meeting_vars) >= 1)

    # Create solver and solve
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 60.0  # 1 minute timeout

    print("Solving league scheduling problem...")
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        print(f"Solution found! Status: {'OPTIMAL' if status == cp_model.OPTIMAL else 'FEASIBLE'}")
        print(f"Solve time: {solver.WallTime():.2f} seconds")

        # Extract and display the solution
        schedule = {}
        for week in weeks:
            schedule[week] = {pod: [] for pod in pods}
            for team in teams:
                for pod in pods:
                    if solver.Value(x[team, week, pod]) == 1:
                        schedule[week][pod].append(team)

        # Display the schedule
        pod_names = ["Early", "Middle", "Late"]
        print("\n" + "="*60)
        print("LEAGUE SCHEDULE")
        print("="*60)

        for week in weeks:
            print(f"\nWeek {week + 1}:")
            print("-" * 20)
            for pod in pods:
                teams_in_pod = schedule[week][pod]
                print(f"{pod_names[pod]:6}: Teams {', '.join(map(str, teams_in_pod))}")

        # Verify constraints
        print("\n" + "="*60)
        print("CONSTRAINT VERIFICATION")
        print("="*60)

        # Check pod usage for each team
        print("\nPod usage by team:")
        for team in teams:
            pod_counts = [0, 0, 0]
            for week in weeks:
                for pod in pods:
                    if solver.Value(x[team, week, pod]) == 1:
                        pod_counts[pod] += 1
            print(f"Team {team:2}: Early={pod_counts[0]}, Middle={pod_counts[1]}, Late={pod_counts[2]}")

        # Check which teams play each other
        print("\nTeam matchups:")
        matchups = defaultdict(set)
        for week in weeks:
            for pod in pods:
                teams_in_pod = schedule[week][pod]
                for team1, team2 in itertools.combinations(teams_in_pod, 2):
                    matchups[team1].add(team2)
                    matchups[team2].add(team1)

        all_pairs_met = True
        for team1 in teams:
            for team2 in teams:
                if team1 < team2:
                    if team2 not in matchups[team1]:
                        print(f"WARNING: Team {team1} never plays Team {team2}")
                        all_pairs_met = False

        if all_pairs_met:
            print("✓ All teams play each other at least once")

        # Summary statistics
        print(f"\nSummary:")
        print(f"✓ Total weeks: {num_weeks}")
        print(f"✓ Total teams: {num_teams}")
        print(f"✓ Teams per pod: {teams_per_pod}")
        print(f"✓ Each team plays {num_weeks} games")

        return schedule

    else:
        print(f"No solution found. Status: {status}")
        return None


schedule = solve_league_scheduling()

Solving league scheduling problem...
Solution found! Status: OPTIMAL
Solve time: 2.37 seconds

LEAGUE SCHEDULE

Week 1:
--------------------
Early : Teams 0, 1, 2, 9
Middle: Teams 4, 6, 10, 11
Late  : Teams 3, 5, 7, 8

Week 2:
--------------------
Early : Teams 0, 1, 2, 9
Middle: Teams 4, 6, 10, 11
Late  : Teams 3, 5, 7, 8

Week 3:
--------------------
Early : Teams 3, 8, 10, 11
Middle: Teams 0, 1, 5, 7
Late  : Teams 2, 4, 6, 9

Week 4:
--------------------
Early : Teams 4, 5, 6, 7
Middle: Teams 0, 1, 3, 8
Late  : Teams 2, 9, 10, 11

Week 5:
--------------------
Early : Teams 5, 7, 10, 11
Middle: Teams 2, 3, 8, 9
Late  : Teams 0, 1, 4, 6

Week 6:
--------------------
Early : Teams 3, 4, 6, 8
Middle: Teams 2, 5, 7, 9
Late  : Teams 0, 1, 10, 11

CONSTRAINT VERIFICATION

Pod usage by team:
Team  0: Early=2, Middle=2, Late=2
Team  1: Early=2, Middle=2, Late=2
Team  2: Early=2, Middle=2, Late=2
Team  3: Early=2, Middle=2, Late=2
Team  4: Early=2, Middle=2, Late=2
Team  5: Early=2, Middle=2,

Week 1:
--------------------
Early : Teams 2, 4, 5, 6
Middle: Teams 0, 7, 10, 11
Late  : Teams 1, 3, 8, 9

Week 2:
--------------------
Early : Teams 2, 4, 10, 11
Middle: Teams 5, 6, 8, 9
Late  : Teams 0, 1, 3, 7

Week 3:
--------------------
Early : Teams 1, 3, 10, 11
Middle: Teams 0, 5, 6, 7
Late  : Teams 2, 4, 8, 9

Week 4:
--------------------
Early : Teams 0, 7, 8, 9
Middle: Teams 1, 2, 3, 4
Late  : Teams 5, 6, 10, 11

Week 5:
--------------------
Early : Teams 1, 3, 5, 6
Middle: Teams 8, 9, 10, 11
Late  : Teams 0, 2, 4, 7

Week 6:
--------------------
Early : Teams 0, 7, 8, 9
Middle: Teams 1, 2, 3, 4
Late  : Teams 5, 6, 10, 11

In [51]:
from ortools.sat.python import cp_model
from collections import defaultdict
import itertools

def solve_league_scheduling_2():
    """
    Solve the league scheduling problem using CP-SAT solver.

    Problem:
    - 12 teams, 3 pods (Early, Middle, Late) with 4 teams each
    - 6 weeks in season
    - Each team plays in exactly one pod per week
    - Each team plays every other team at least once
    - Each team plays in each pod at most twice
    """

    # Problem parameters
    num_teams = 12
    num_pods = 3
    teams_per_pod = 4
    num_weeks = 5

    teams = list(range(num_teams))
    pods = list(range(num_pods))  # 0=Early, 1=Middle, 2=Late
    weeks = list(range(num_weeks))

    # Create the model
    model = cp_model.CpModel()

    # Decision variables: x[team][week][pod] = 1 if team plays in pod during week
    x = {}
    for team in teams:
        for week in weeks:
            for pod in pods:
                x[team, week, pod] = model.NewBoolVar(f'x_{team}_{week}_{pod}')

    # Constraint 1: Each team plays in exactly one pod per week
    for team in teams:
        for week in weeks:
            model.Add(sum(x[team, week, pod] for pod in pods) == 1)

    # Constraint 2: Each pod has exactly 4 teams per week
    for week in weeks:
        for pod in pods:
            model.Add(sum(x[team, week, pod] for team in teams) == teams_per_pod)

    # Constraint 3: Each team plays in each pod at most twice
    for team in teams:
        for pod in pods:
            model.Add(sum(x[team, week, pod] for week in weeks) <= 2)

    # Constraint 4: Each team plays every other team at least once
    # Two teams play each other if they are in the same pod in the same week
    for team1 in teams:
        for team2 in teams:
            if team1 < team2:  # Avoid duplicate constraints
                # Create a variable for each week/pod combination where both teams could meet
                meeting_vars = []
                for week in weeks:
                    for pod in pods:
                        # Both teams are in the same pod in the same week
                        meet_var = model.NewBoolVar(f'meet_{team1}_{team2}_{week}_{pod}')
                        model.AddBoolAnd([x[team1, week, pod], x[team2, week, pod]]).OnlyEnforceIf(meet_var)
                        model.AddBoolOr([x[team1, week, pod].Not(), x[team2, week, pod].Not()]).OnlyEnforceIf(meet_var.Not())
                        meeting_vars.append(meet_var)

                # Teams must meet at least once
                model.Add(sum(meeting_vars) >= 1)

    # Constraint 5: All weekly pod arrangements must be unique
    # For each pair of weeks, at least one pod must have a different composition
    for week1 in weeks:
        for week2 in weeks:
            if week1 < week2:  # Avoid duplicate constraints
                # For each pod, create a variable indicating if the pod composition is identical
                pod_identical_vars = []
                for pod in pods:
                    # Create variables for each team being in this pod in both weeks
                    same_composition_vars = []
                    for team in teams:
                        # Team is in the same pod in both weeks
                        both_in_pod = model.NewBoolVar(f'both_{team}_{week1}_{week2}_{pod}')
                        model.AddBoolAnd([x[team, week1, pod], x[team, week2, pod]]).OnlyEnforceIf(both_in_pod)
                        model.AddBoolOr([x[team, week1, pod].Not(), x[team, week2, pod].Not()]).OnlyEnforceIf(both_in_pod.Not())

                        # Team is not in the pod in either week
                        neither_in_pod = model.NewBoolVar(f'neither_{team}_{week1}_{week2}_{pod}')
                        model.AddBoolAnd([x[team, week1, pod].Not(), x[team, week2, pod].Not()]).OnlyEnforceIf(neither_in_pod)
                        model.AddBoolOr([x[team, week1, pod], x[team, week2, pod]]).OnlyEnforceIf(neither_in_pod.Not())

                        # Team's status is the same in both weeks for this pod
                        same_status = model.NewBoolVar(f'same_{team}_{week1}_{week2}_{pod}')
                        model.AddBoolOr([both_in_pod, neither_in_pod]).OnlyEnforceIf(same_status)
                        model.AddBoolAnd([both_in_pod.Not(), neither_in_pod.Not()]).OnlyEnforceIf(same_status.Not())

                        same_composition_vars.append(same_status)

                    # Pod has identical composition if all teams have the same status
                    pod_identical = model.NewBoolVar(f'pod_identical_{week1}_{week2}_{pod}')
                    model.AddBoolAnd(same_composition_vars).OnlyEnforceIf(pod_identical)
                    model.AddBoolOr([var.Not() for var in same_composition_vars]).OnlyEnforceIf(pod_identical.Not())
                    pod_identical_vars.append(pod_identical)

                # At least one pod must have different composition between the two weeks
                model.AddBoolOr([var.Not() for var in pod_identical_vars])

    # Create solver and solve
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 60.0  # 1 minute timeout

    print("Solving league scheduling problem...")
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        print(f"Solution found! Status: {'OPTIMAL' if status == cp_model.OPTIMAL else 'FEASIBLE'}")
        print(f"Solve time: {solver.WallTime():.2f} seconds")

        # Extract and display the solution
        schedule = {}
        for week in weeks:
            schedule[week] = {pod: [] for pod in pods}
            for team in teams:
                for pod in pods:
                    if solver.Value(x[team, week, pod]) == 1:
                        schedule[week][pod].append(team)

        # Display the schedule
        pod_names = ["Early", "Middle", "Late"]
        print("\n" + "="*60)
        print("LEAGUE SCHEDULE")
        print("="*60)

        for week in weeks:
            print(f"\nWeek {week + 1}:")
            print("-" * 20)
            for pod in pods:
                teams_in_pod = schedule[week][pod]
                print(f"{pod_names[pod]:6}: Teams {', '.join(map(str, teams_in_pod))}")

        # Verify constraints
        print("\n" + "="*60)
        print("CONSTRAINT VERIFICATION")
        print("="*60)

        # Check pod usage for each team
        print("\nPod usage by team:")
        for team in teams:
            pod_counts = [0, 0, 0]
            for week in weeks:
                for pod in pods:
                    if solver.Value(x[team, week, pod]) == 1:
                        pod_counts[pod] += 1
            print(f"Team {team:2}: Early={pod_counts[0]}, Middle={pod_counts[1]}, Late={pod_counts[2]}")

        # Check which teams play each other
        print("\nTeam matchups:")
        matchups = defaultdict(set)
        for week in weeks:
            for pod in pods:
                teams_in_pod = schedule[week][pod]
                for team1, team2 in itertools.combinations(teams_in_pod, 2):
                    matchups[team1].add(team2)
                    matchups[team2].add(team1)

        all_pairs_met = True
        for team1 in teams:
            for team2 in teams:
                if team1 < team2:
                    if team2 not in matchups[team1]:
                        print(f"WARNING: Team {team1} never plays Team {team2}")
                        all_pairs_met = False

        if all_pairs_met:
            print("✓ All teams play each other at least once")

        # Summary statistics
        print(f"\nSummary:")
        print(f"✓ Total weeks: {num_weeks}")
        print(f"✓ Total teams: {num_teams}")
        print(f"✓ Teams per pod: {teams_per_pod}")
        print(f"✓ Each team plays {num_weeks} games")

        return schedule

    else:
        print(f"No solution found. Status: {status}")
        return None

schedule_2 = solve_league_scheduling_2()

Solving league scheduling problem...
Solution found! Status: OPTIMAL
Solve time: 0.37 seconds

LEAGUE SCHEDULE

Week 1:
--------------------
Early : Teams 0, 1, 4, 11
Middle: Teams 2, 3, 6, 8
Late  : Teams 5, 7, 9, 10

Week 2:
--------------------
Early : Teams 0, 2, 6, 11
Middle: Teams 1, 4, 9, 10
Late  : Teams 3, 5, 7, 8

Week 3:
--------------------
Early : Teams 1, 4, 5, 7
Middle: Teams 2, 6, 9, 10
Late  : Teams 0, 3, 8, 11

Week 4:
--------------------
Early : Teams 3, 8, 9, 10
Middle: Teams 0, 5, 7, 11
Late  : Teams 1, 2, 4, 6

Week 5:
--------------------
Early : Teams 2, 5, 6, 7
Middle: Teams 1, 3, 4, 8
Late  : Teams 0, 9, 10, 11

CONSTRAINT VERIFICATION

Pod usage by team:
Team  0: Early=2, Middle=1, Late=2
Team  1: Early=2, Middle=2, Late=1
Team  2: Early=2, Middle=2, Late=1
Team  3: Early=1, Middle=2, Late=2
Team  4: Early=2, Middle=2, Late=1
Team  5: Early=2, Middle=1, Late=2
Team  6: Early=2, Middle=2, Late=1
Team  7: Early=2, Middle=1, Late=2
Team  8: Early=1, Middle=2, L